Data label manual

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder

In [7]:
#LOAD DATA (DATA SUDAH DILABELI MANUAL)
df = pd.read_csv("/content/drive/MyDrive/Ulasan Webtoon.csv")

In [12]:
# =====================================================
# 2. PISAHKAN DATA MANUAL MENJADI 100 POSITIF, 100 NEGATIF, 50 NETRAL
# =====================================================

# Ensure 'label' column is numeric for mapping, handling potential string representations of numbers
df['numeric_label'] = pd.to_numeric(df['label'], errors='coerce')

# Map numerical labels (from 1-5 scale) to categorical strings for selection purposes
def map_sentiment_for_selection(score):
    if score in [4, 5]: return 'positif'
    if score in [1, 2]: return 'negatif'
    if score == 3: return 'netral'
    return None # For NaN or other unexpected values

df['temp_categorical_label'] = df['numeric_label'].apply(map_sentiment_for_selection)

# Define target sample counts
target_pos_count = 100
target_neg_count = 100
target_netral_count = 50

# Filter data for each category using the new temporary categorical label
df_pos_candidates = df[df['temp_categorical_label'] == 'positif']
df_neg_candidates = df[df['temp_categorical_label'] == 'negatif']
df_netral_candidates = df[df['temp_categorical_label'] == 'netral']

# Sample data for each category, ensuring not to request more samples than available
actual_pos_count = min(target_pos_count, len(df_pos_candidates))
df_pos = df_pos_candidates.sample(actual_pos_count, random_state=42)
if actual_pos_count < target_pos_count:
    print(f"Warning: Only {actual_pos_count} 'positif' samples available, requested {target_pos_count}.")

actual_neg_count = min(target_neg_count, len(df_neg_candidates))
df_neg = df_neg_candidates.sample(actual_neg_count, random_state=42)
if actual_neg_count < target_neg_count:
    print(f"Warning: Only {actual_neg_count} 'negatif' samples available, requested {target_neg_count}.")

actual_netral_count = min(target_netral_count, len(df_netral_candidates))
df_netral = df_netral_candidates.sample(actual_netral_count, random_state=42)
if actual_netral_count < target_netral_count:
    print(f"Warning: Only {actual_netral_count} 'netral' samples available, requested {target_netral_count}.")

# Map categorical labels to numeric (0, 1, 2) for final 'label' column in df_manual
def map_sentiment_to_numeric(categorical_label):
    if categorical_label == 'negatif': return 0
    if categorical_label == 'netral': return 1
    if categorical_label == 'positif': return 2
    return -1 # Should not happen for manually labeled data

# Concatenate the manually selected samples and create the final 'label' column
df_manual_selected = pd.concat([df_pos, df_neg, df_netral])
df_manual_selected['label'] = df_manual_selected['temp_categorical_label'].apply(map_sentiment_to_numeric)
df_manual = df_manual_selected[['ulasan', 'label']].copy() # Keep only ulasan and the new numeric label

# =====================================================
# 3. SIAPKAN DATA UNLABELED (SISA UNTUK SELF-TRAINING)
# =====================================================
# Get the indices of the selected manual samples from the original df
manual_selected_indices = pd.Index(list(df_pos.index) + list(df_neg.index) + list(df_netral.index))

# Drop the selected manual samples from the original df to get unlabeled data
df_unlabeled = df.drop(manual_selected_indices).copy()
# Set the 'label' for unlabeled data to -1, and keep only 'ulasan' and 'label'
df_unlabeled['label'] = -1
df_unlabeled = df_unlabeled[['ulasan', 'label']].copy()

# =====================================================
# 4. GABUNGKAN DATA MANUAL + UNLABELED
# =====================================================
df_all = pd.concat([df_manual, df_unlabeled]).reset_index(drop=True)

# Drop temporary columns from the original df if they are no longer needed
df = df.drop(columns=['numeric_label', 'temp_categorical_label'], errors='ignore')

# Display some info about the created dataframes for verification
print(f"df_manual shape: {df_manual.shape}")
print(f"df_manual label value counts:\n{df_manual['label'].value_counts()}")
print(f"df_unlabeled shape: {df_unlabeled.shape}")
print(f"df_unlabeled label value counts:\n{df_unlabeled['label'].value_counts()}")
print(f"df_all shape: {df_all.shape}")
print(f"df_all label value counts:\n{df_all['label'].value_counts()}")

df_manual shape: (250, 2)
df_manual label value counts:
label
2    100
0    100
1     50
Name: count, dtype: int64
df_unlabeled shape: (9750, 2)
df_unlabeled label value counts:
label
-1    9750
Name: count, dtype: int64
df_all shape: (10000, 2)
df_all label value counts:
label
-1    9750
 2     100
 0     100
 1      50
Name: count, dtype: int64


In [13]:
#TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df_all['ulasan'])

# Label encoding
le = LabelEncoder()
y = df_all['label'].replace(-1, -1)   # tetap -1 untuk unlabeled

# Encode hanya label manual
y_encoded = y.copy()
mask_manual = y_encoded != -1
y_encoded[mask_manual] = le.fit_transform(y_encoded[mask_manual])

In [15]:
# 6. SELF TRAINING (SEMISUPERVISED)
base_model = MultinomialNB()

self_training_model = SelfTrainingClassifier(
    base_model,
    threshold=0.8,     # model harus yakin ≥ 80%
    verbose=True
)

self_training_model.fit(X, y_encoded)

End of iteration 1, added 34 new labels.
End of iteration 2, added 1010 new labels.
End of iteration 3, added 8516 new labels.
End of iteration 4, added 185 new labels.


SelfTrainingClassifier(estimator=MultinomialNB(), threshold=0.8, verbose=True)

In [16]:
#DAPATKAN SELF-LABEL UNTUK DATA UNLABELED
predicted_labels = self_training_model.predict(X)

# Kembalikan label ke nama kategori
decoded_labels = pd.Series(predicted_labels).replace(
    dict(zip(range(len(le.classes_)), le.classes_))
)

df_all['label_final'] = decoded_labels

In [17]:
#PEMISAHAN DATA MANUAL + SELF-TRAINED
df_final_manual = df_all.loc[df_manual.index]
df_final_self = df_all.loc[df_unlabeled.index]

In [18]:
#SIMPAN DATA
df_final_manual.to_csv("data_manual_250.csv", index=False)
df_final_self.to_csv("data_self_training_750.csv", index=False)
df_all.to_csv("data_final_1000.csv", index=False)

print("Selesai! File berhasil dibuat:")
print("- data_manual_250.csv (250 data manual)")
print("- data_self_training_750.csv (750 data hasil self-training)")
print("- data_final_1000.csv (gabungan)")

Selesai! File berhasil dibuat:
- data_manual_250.csv (250 data manual)
- data_self_training_750.csv (750 data hasil self-training)
- data_final_1000.csv (gabungan)
